# SleepPath Research Lab · Colab v0.1
## De una heurística clínica a un modelo pronóstico con validación

**Objetivo:** prototipar una metodología reproducible para identificar *fenotipos de sueño* y estudiar su asociación con desenlaces cardiovasculares/mortalidad. La versión HTML **no** calcula riesgos absolutos; este cuaderno es el primer paso para construir y validar un modelo real.

> ⚠️ **ESTRICTAMENTE INVESTIGACIÓN.** El modo inicial `DEMO` genera datos **totalmente sintéticos**; cualquier coeficiente, métrica, curva o estrato resultante de ellos es ficticio y NO puede orientar a pacientes. La estratificación pronóstica real requiere cohortes apropiadas, endpoints adjudicados, exclusión de desenlaces prevalentes, autorizaciones y validación externa.

**Inspiración**: [Bilal et al., Nat Commun 2026](https://doi.org/10.1038/s41467-026-75326-9) (embeddings fisiológicos y clústeres), [Thapa et al., Nat Med 2026 (SleepFM)](https://doi.org/10.1038/s41591-025-04133-4) ([código](https://github.com/zou-group/sleepfm-clinical)), [Windred et al. 2024](https://doi.org/10.1093/sleep/zsad253) (regularidad) y [SHHS/NSRR](https://sleepdata.org/datasets/shhs) (cohorte con PSG y seguimiento).

In [ ]:
# Celda 1 — Ejecutar en Google Colab (no requiere GPU para este baseline tabular)
%pip -q install "lifelines>=0.30,<0.32" "scikit-learn>=1.4,<1.8" pandas matplotlib seaborn


## 1. Diccionario analítico estandarizado

Para cargar un archivo real, suba un CSV `sleep_cohort.csv` en `/content/`. **No** se pretende que SHHS use literalmente estos nombres; deberá mapearse su diccionario y sus unidades.

Campos necesarios:

| Campo | Tipo / descripción |
| --- | --- |
| `patient_id` | identificador anonimizado para partición por paciente |
| `time_days` | días desde PSG/índice hasta desenlace o censura |
| `event` | 1 si ocurre **un desenlace especificado** (p. ej. muerte por cualquier causa), 0 censura |
| `age` | años al inicio |
| `sleep_hours` | horas de sueño registradas (preferible PSG/actigrafía) |
| `sleep_efficiency` | porcentaje 0–100 |
| `ahi` | índice apnea/hipopnea, eventos/h |

Campos predictivos **opcionales:** `bmi`, `isi`, `ess`, `wake_sd_min`, `stop_bang`, `comorbidity`, `sex_binary`.

**Prioridad epidemiológica:** antes de ajustar el modelo, definir un desenlace clínico primario y una ventana. Para incidentes cardiovasculares, excluir eventos previos y documentar muerte como riesgo competidor cuando corresponda; para mortalidad global no aplica muerte como competidor. Mantener todos los predictores medidos **antes** del desenlace y no agregar información de seguimiento al baseline.


In [ ]:
# Celda 2 — DEMO sintética o carga de CSV con datos reales
import numpy as np, pandas as pd
from pathlib import Path
MODE = "DEMO"  # Cambiar a "REAL" SOLO después de armonizar cohorte, permisos y endpoints.
RNG = np.random.default_rng(20261027)
if MODE == "DEMO":
    n=950
    age=RNG.uniform(30,83,n)
    bmi=np.clip(RNG.normal(28,5,n),18,45)
    sleep_hours=np.clip(RNG.normal(6.9,1.25,n),3.5,10)
    efficiency=np.clip(RNG.normal(82,10,n),45,99)
    ahi=np.clip(RNG.gamma(1.3,12,n),0,90)
    wake_sd_min=np.clip(RNG.gamma(1.8,25,n),2,180)
    ess=np.clip(RNG.normal(9,4,n),0,24)
    isi=np.clip(RNG.normal(10,6,n),0,28)
    comorbidity=RNG.binomial(1,0.32,n)
    sex_binary=RNG.binomial(1,0.48,n)
    # Esta ecuación genera datos artificiales; NO representa hazard clínico real.
    latent=(0.04*(age-55)+0.03*(bmi-28)+0.42*(sleep_hours<6)+
            0.011*(wake_sd_min-40)+0.4*comorbidity+0.004*ahi)
    event_time=RNG.exponential(scale=np.exp(-latent)/0.00030)
    censor_time=RNG.uniform(1200,3000,n)
    d=pd.DataFrame({'patient_id': [f'SYN-{i:04d}' for i in range(n)],
          'time_days':np.minimum(event_time,censor_time),
          'event':(event_time<=censor_time).astype(int),
          'age':age,'bmi':bmi,'sleep_hours':sleep_hours,
          'sleep_efficiency':efficiency,'ahi':ahi,
          'wake_sd_min':wake_sd_min,'isi':isi,'ess':ess,
          'comorbidity':comorbidity,'sex_binary':sex_binary})
    print('⚠️ DEMOSTRACIÓN: LOS DATOS Y TODOS LOS OUTCOMES SON SINTÉTICOS')
else:
    file=Path('/content/sleep_cohort.csv')
    assert file.exists(), 'Suba sleep_cohort.csv a /content después de armonizar y anonimizar datos.'
    d=pd.read_csv(file)
    print('Cohorte real seleccionada. Verifique consentimiento/DUA, diccionario y privacidad.')
print('Filas:',len(d),'\nTasa de eventos en el dataset:', round(d['event'].mean(),3))
d.head()


## 2. Control de calidad y separación por paciente

No eliminar silenciosamente datos con problemas: este baseline emite alertas. El imputador y el escalador se ajustan **solo con datos de entrenamiento** para prevenir fuga de información. El objetivo se modela como supervivencia censurada y no como clasificación binaria naïve.


In [ ]:
# Celda 3 — QC y partición holdout (pacientes, no registros)
from sklearn.model_selection import GroupShuffleSplit
required=['patient_id','time_days','event','age','sleep_hours','sleep_efficiency','ahi']
missing=[c for c in required if c not in d.columns]
assert not missing, f'Faltan campos requeridos: {missing}'
assert d['event'].dropna().isin([0,1]).all(), 'event debe ser 0/1'
assert (d['time_days']>0).all(), 'time_days debe ser >0'
assert d['patient_id'].notna().all(), 'patient_id no puede estar vacío'
assert ((d['sleep_hours'].dropna()>=0)&(d['sleep_hours'].dropna()<=16)).all(), 'Sueño fuera de 0–16 horas'
assert ((d['sleep_efficiency'].dropna()>=0)&(d['sleep_efficiency'].dropna()<=100)).all(), 'Eficiencia fuera de 0–100%'
features=[c for c in ['age','bmi','sleep_hours','sleep_efficiency','ahi','wake_sd_min',
                    'isi','ess','stop_bang','comorbidity','sex_binary'] if c in d.columns]
d=d.copy()
for col in features+['time_days','event']: d[col]=pd.to_numeric(d[col],errors='coerce')
assert d[['time_days','event']].notna().all().all(), 'Faltan desenlaces o tiempos'
features=[c for c in features if d[c].notna().any()]
idx_train,idx_test=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42).split(d,groups=d['patient_id']))
train=d.iloc[idx_train].reset_index(drop=True)
test=d.iloc[idx_test].reset_index(drop=True)
assert set(train.patient_id).isdisjoint(set(test.patient_id)), 'Fuga entre pacientes'
print(f'Entrenamiento: {len(train)} ({train.event.sum()} eventos) | Test: {len(test)} ({test.event.sum()} eventos)')
print('Missing % train:\n', (train[features].isna().mean()*100).round(1).to_string())
assert train.event.sum()>=20, 'Muy pocos eventos para este ejemplo: replantee diseño y variables.'


## 3. Prototipado pronóstico (baseline Cox regularizado)

Estimación relativa de *hazard* con penalización, **no** probabilidad clínica calibrada automáticamente. Para despliegue real: validación temporal, bootstrap interno, calibración a 2–5 años, evaluación de censura, Schoenfeld/proporcionalidad, competing risks y evaluación externa por centro/edad/sexo. El C-index mide discriminación, **no** calibración ni utilidad clínica.


In [ ]:
# Celda 4 — Transformación entrenada sin mirar el test
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
prep=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
Xtr=pd.DataFrame(prep.fit_transform(train[features]),columns=features)
Xte=pd.DataFrame(prep.transform(test[features]),columns=features)
tr_cox=Xtr.assign(time_days=train.time_days.to_numpy(),event=train.event.to_numpy())
model=CoxPHFitter(penalizer=0.5)
model.fit(tr_cox,duration_col='time_days',event_col='event')
pred=model.predict_partial_hazard(Xte).to_numpy().ravel()
cstat=concordance_index(test.time_days,-pred,test.event)
print('⚠️ DATOS SINTÉTICOS: rendimiento simulado, nunca clínico.' if MODE=='DEMO' else 'C-index holdout (requiere validación independiente):')
print(f'C-index: {cstat:.3f} | test n={len(test)} | eventos={int(test.event.sum())}')
model.summary[['coef','exp(coef)','p']].round(3)


## 4. Fenotipos de sueño no supervisados y outcomes

Esta etapa **no** atribuye los cinco fenotipos de Bilal et al. a datos tabulares. Ajusta K-means *solo en entrenamiento* y describe asociaciones observacionales en el holdout. La supervivencia según grupo puede estar confundida por edad, comorbilidades y selección; debe confirmarse con Cox ajustado/validación externa en análisis principal.


In [ ]:
# Celda 5 — Fenotipos K-means y Kaplan-Meier holdout
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
candidate=[]
for k in range(2,6):
    km=KMeans(n_clusters=k,random_state=42,n_init=15)
    labels=km.fit_predict(Xtr)
    sizes=np.bincount(labels)/len(labels)
    silhouette=silhouette_score(Xtr,labels)
    candidate.append((k,silhouette,sizes.min()))
print(pd.DataFrame(candidate,columns=['k','silhouette','fracción_mínima_cluster']).round(3))
# k=3 se usa solo para visualizar el ejemplo; NO está validado.
cluster_model=KMeans(n_clusters=3,random_state=42,n_init=15).fit(Xtr)
group=cluster_model.predict(Xte)
fig,ax=plt.subplots(figsize=(8,5))
for z in sorted(np.unique(group)):
    subset=test.loc[group==z]
    if len(subset)==0:continue
    KaplanMeierFitter().fit(subset.time_days,subset.event,label=f'Clúster exploratorio {z} (n={len(subset)})').plot_survival_function(ax=ax)
ax.set(xlabel='Días desde inicio',ylabel='Supervivencia observada',title='KM exploratoria — '+('DATOS SINTÉTICOS' if MODE=='DEMO' else 'COHORTE REAL (NO AJUSTADA)'))
ax.grid(alpha=.2)
plt.show()


## 5. Validación imprescindible antes de cualquier predicción clínica

**Separar tres preguntas científicas diferentes:**

1. **Predicción de outcomes:** ¿mejora C-index/AUC temporal, calibración a 2/5 años, Brier IPCW y utilidad neta (decision-curve) frente a edad, comorbilidades, AHI e índices existentes? Probar subgrupos y reentrenamiento/validación externa temporal/por centro.
2. **Fenotipos reproducibles:** ¿clústeres estables en bootstrap, cohortes nuevas, mediciones repetidas, con evidencia de asociación independiente a resultados? Un silhouette alto no basta.
3. **Personalización terapéutica:** ¿el fenotipo modifica el **efecto causal** de CBT-I, PAP, luz o intervención de horarios? Una asociación pronóstica **NO** identifica quién se beneficiará. Requiere cohortes con tratamiento y sesgo controlado o, preferiblemente, ensayos estratificados.

**Fuentes de datos posibles:** [SHHS](https://sleepdata.org/datasets/shhs) (PSG + desenlaces), [MESA Sleep](https://sleepdata.org/datasets/mesa), [MrOS](https://sleepdata.org/datasets/mros), cohorte local con ISI, ESS, diarios/actigrafía y seguimiento. Revisar acuerdos de uso y diccionarios; no mezclar escalas ni usar eventos prevalentes como incidentes.

**Análisis de equidad y aplicabilidad:** verificar faltantes no aleatorios, diferencias por edad, sexo, centros y poblaciones mexicanas; estudiar *calibration drift*. Los HR de los papers no deben copiarse como pesos de un score local.


In [ ]:
# Celda 6 — Plantilla de exportación SEGURA, sin desplegar un modelo no validado
report={
    'mode':MODE,
    'training_records':len(train),
    'test_records':len(test),
    'test_events':int(test.event.sum()),
    'holdout_concordance_exploratory':round(float(cstat),3),
    'data_schema_version':'sleeppath-cohort-v0.1',
    'validated_for_clinical_use':False,
    'not_for_patient_use':True,
    'next_steps':['Endpoint adjudicado y horizonte','Validación temporal/externa',
                  'Calibración y Brier IPCW','Revisión de sesgo','Análisis causal de respuesta al tratamiento']
}
import json
print(json.dumps(report,ensure_ascii=False,indent=2))
# Por seguridad no exportamos coeficientes ni predicciones individuales para insertar en HTML.
